In [1]:
# 필요한 라이브러리들을 임포트합니다.
from sklearn.linear_model import LogisticRegression # 로지스틱 회귀 모델
from sklearn.ensemble import RandomForestClassifier # 랜덤 포레스트 분류기
from sklearn.model_selection import train_test_split, GridSearchCV # 훈련/테스트 데이터 분할 및 그리드 서치를 위한 모듈
from sklearn.feature_selection import SelectKBest, VarianceThreshold, f_classif # 특성 선택을 위한 모듈 (최고 K개 선택, 분산 임계값, ANOVA F-값)
from sklearn.tree import DecisionTreeClassifier # 결정 트리 분류기
from sklearn.metrics import roc_auc_score, fbeta_score, make_scorer # ROC AUC 점수, F-베타 점수, 커스텀 스코어러 생성
from xgboost import XGBClassifier # XGBoost 분류기
import shap # SHAP(SHapley Additive exPlanations) 라이브러리 (모델 예측 설명)
import matplotlib.pyplot as plt # 데이터 시각화를 위한 라이브러리

import pandas as pd # 데이터 조작 및 분석을 위한 라이브러리
import numpy as np # 수치 계산을 위한 라이브러리
import datetime as dt # 날짜 및 시간 처리를 위한 라이브러리
import json # JSON 데이터 처리를 위한 라이브러리

In [2]:
# 경고 메시지 처리를 위한 모듈
import warnings 

# 'use_label_encoder' 경고만 무시합니다.
warnings.filterwarnings("ignore")

#### prepare "data/initial_dataset.p"

In [3]:
# # .xlsx -> .p : 1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with 3Lots_FT1_FT2_FT3

# # 파일 경로 지정
# file_path = 'data/1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with 3Lots_FT1_FT2_FT3.xlsx'

# # 엑셀 파일을 DataFrame으로 읽어오기
# # 기본적으로 첫 번째 시트를 읽어옵니다.
# data_row = pd.read_excel(file_path)

# # Define the file path
# output_file_path = 'data/1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with 3Lots_FT1_FT2_FT3.p'

# # Save the DataFrame to a pickle file
# data_row.to_pickle(output_file_path)

In [4]:
# # .xlsx -> .p : 1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with QualWfrs_FT&QAwithQAAfterStress

# # 파일 경로 지정
# file_path = 'data\\1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with QualWfrs_FT&QAwithQAAfterStress.xlsx'

# # 엑셀 파일을 DataFrame으로 읽어오기
# # 기본적으로 첫 번째 시트를 읽어옵니다.
# data_row = pd.read_excel(file_path)

# # Define the file path
# output_file_path = 'data\\1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with QualWfrs_FT&QAwithQAAfterStress.p'

# # Save the DataFrame to a pickle file
# data_row.to_pickle(output_file_path)

In [5]:
# read *.p
pickle_file_path_1 = 'data\\1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with 3Lots_FT1_FT2_FT3.p'
data_row_1 = pd.read_pickle(pickle_file_path_1)
pickle_file_path_2 = 'data\\1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with QualWfrs_FT&QAwithQAAfterStress.p'
data_row_2 = pd.read_pickle(pickle_file_path_2)

In [6]:
# 필요한 컬럼만 keep

# 파일 경로 지정
file_path = 'data/cols_to_keep.csv'

# CSV 파일을 DataFrame으로 읽어오기
cols_to_keep_df = pd.read_csv(file_path)

cols_to_keep = cols_to_keep_df.iloc[:, 0].tolist()

data_row_1 = data_row_1[cols_to_keep]

In [7]:
# data_row <= data_row1 data_row2

# data_row_2에서 조인할 컬럼만 선택
columns_to_join = ['DevID of 1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP',
                  #  'wafer_id', 
                #    'SensorOffsetHot-RoomAfterBake', 
                #    'SensorOffsetHot-ColdAfterBake', 
                   'BG pass/fail']

# 선택한 컬럼으로 data_row_2의 부분집합 DataFrame 생성
data_row_2_subset = data_row_2[columns_to_join]

# data_row_1에 data_row_2의 선택된 컬럼들을 조인 키 'DevID'로 병합
data_row = pd.merge(data_row_1, data_row_2_subset, on='DevID of 1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP', how='left')

# # 결과 DataFrame 확인
# print(merged_df.head())

In [8]:
initial_dataset = data_row.copy() # 원본 데이터셋 복사
# processed_dataset = initial_dataset.copy() # 원본 데이터셋 복사

In [9]:
# prepare for target

initial_dataset['Pass/Fail_pass'] = ((initial_dataset['soft_bin of FT1'] == 1) &
                   (initial_dataset['soft_bin of FT2'] == 1) &
                   (initial_dataset['soft_bin'] == 1)).astype(int)

In [10]:
# prepare for base model

initial_dataset['band gap dpat'] = initial_dataset['BG pass/fail'].apply(lambda x: 'bandGapFail' if x == 'impossible wafer' else 'ok for band gap')

# 컬럼 이름 변경 딕셔너리 생성
new_column_names = {
    'wafer_id': 'WAFER_NO',
    'DevID of 1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP': 'DevID'
}

# .rename() 메서드를 사용하여 컬럼 이름 변경 (inplace=True로 원본 데이터프레임에 바로 적용)
initial_dataset.rename(columns=new_column_names, inplace=True)

# 변경된 컬럼 이름 확인
# print(initial_dataset.columns)

In [11]:
# initial_dataset

In [12]:
# 제거할 컬럼 리스트 정의
columns_to_drop = [
    'soft_bin of FT1',
    'soft_bin of FT2',
    'soft_bin',
    'BG pass/fail'
]

# 컬럼 drop (원본 DataFrame을 변경하려면 inplace=True 사용)
# 또는 새로운 DataFrame을 만들려면 processed_dataset = processed_dataset.drop(...) 사용
initial_dataset.drop(columns=columns_to_drop, inplace=True)

In [13]:
# Rename the columns
initial_dataset.rename(columns={'x_pos': 'X', 'y_pos': 'Y'}, inplace=True)

In [14]:
# Radius 컬럼 계산
# np.sqrt() 함수는 각 요소의 제곱근을 계산합니다.
initial_dataset['Radius'] = np.sqrt(initial_dataset['X']**2 + initial_dataset['Y']**2)

In [15]:
initial_dataset.to_pickle("data/initial_dataset.p")

#### scnarios

In [16]:
# import vars and function

from algos.algos import *
from config.config import *

In [17]:
import copy

In [18]:
##### preprocess_dataset
# def preprocess_dataset(initial_dataset: pd.DataFrame):
    # return processed_dataset # 전처리된 데이터셋 반환

preprocessed_dataset = preprocess_dataset(initial_dataset)

##### create_train_and_test_data
# def create_train_test_data(
#     preprocessed_dataset: pd.DataFrame,
#     split_parameter: dict = None
# ):
#     return train_data, test_data, split_parameter_info

split_parameter = copy.deepcopy(split_parameter_default)
train_data, test_data, split_parameter_info = create_train_test_data(preprocessed_dataset, split_parameter)




     데이터셋 전처리 중...
     전처리 완료!



##############################################################################################################################
# 3) Create Train/Test Split (훈련/테스트 데이터 분할) 
##############################################################################################################################

     훈련 및 테스트 데이터셋 생성 중...
     - 분할 전 필터링 미적용.
     - Feature Generation 미적용.

     - 분할 전 훈련 데이터 클래스 분포: {0.0: 3546, 1.0: 71}
     - 샘플링 미적용


In [19]:
# ### senario sample

# ##### train_model_*logistic_regression*
# # def train_model_logistic_regression(train_dataset: pd.DataFrame, feature_selection_info: dict, train_parameters: dict = None):
#     # return model_fitted, importance, model_parameter_info
# trained_model_logistic_regression, feature_importance_logistic_regression, train_parameters_info_logistic_regression = train_model_logistic_regression(train_data, feature_selection_info_var, train_parameters_list_default["random_forest"])

# ##### predict_the_test_data_*logistic_regression*
# # def forecast(test_dataset: pd.DataFrame, trained_model, feature_selection_info: dict):
#     # return predictions, [shap_values, X]
# forecast_dataset_logistic_regression, shap_values_logistic_regression = forecast(test_data, trained_model_logistic_regression, feature_selection_info_var)

# ##### find_best_threshold_*logistic_regression*
# # def find_best_threshold(best_model, train_dataset, feature_selection_info: dict):
#     # return train_dataset, best_threshold
# train_dataset_proba_logistic_regression, best_threshold_logistic_regression = find_best_threshold(trained_model_logistic_regression, train_data, feature_selection_info_var)

# ##### task_roc_logistic_*regression*
# # def roc_from_scratch(probabilities, test_dataset, partitions=100):
#     # return roc_data, auc_score
# roc_data_logistic_regression, auc_score_logistic_regression = roc_from_scratch(forecast_dataset_logistic_regression, test_data, partitions=100)

# ##### create_metrics_on_train_*logistic_regression*
# #def create_metrics_on_train(train_dataset, threshold): 
# #   return train_dataset
# train_dataset_metrics_logistic_regression = create_metrics_on_train(train_dataset_proba_logistic_regression, best_threshold_logistic_regression)

# ##### task_create_metrics_logistic_regression
# # def create_metrics(
# #     predictions: np.array, test_dataset: pd.DataFrame, auc_score, threshold
# # ):
#     # return metrics   
# metrics_logistic_regression = create_metrics(forecast_dataset_logistic_regression, test_data, auc_score_logistic_regression, best_threshold_logistic_regression)

# ##### task_create_results_logistic_regression
# # def create_results(forecast_values, test_dataset, threshold):
#     # return results
# results_logistic_regression = create_results(forecast_dataset_logistic_regression, test_data, best_threshold_logistic_regression)

##### 사용자가 지정한 임계값 활용 - 변수선택 및 변수중요도 계산

In [20]:
##### select_feature
# def select_feature(train_data: pd.DataFrame, feature_selector_params: Dict) -> Dict:
    # return feature_selection_info
feature_selector_params_var = copy.deepcopy(feature_selector_params_FeatureFilter_default)
feature_selector_params_var["filter_methods"]["apply_variance_filter"] = True
feature_selector_params_var["filter_methods"]["var_threshold"] = 0.00
feature_selection_info_var = select_feature(train_data, feature_selector_params_var)

feature_selector_params_licor = copy.deepcopy(feature_selector_params_FeatureFilter_default)
feature_selector_params_licor["filter_methods"]["apply_target_linear_corr_filter"] = True
feature_selector_params_licor["filter_methods"]["target_linear_corr_threshold"] = 0.00
feature_selection_info_licor = select_feature(train_data, feature_selector_params_licor)

feature_selector_params_xicor = copy.deepcopy(feature_selector_params_FeatureFilter_default)
feature_selector_params_xicor["filter_methods"]["apply_target_xicor_filter"] = True
feature_selector_params_xicor["filter_methods"]["target_xicor_threshold"] = 0.00
feature_selection_info_xicor = select_feature(train_data, feature_selector_params_xicor)

feature_selector_params_sfm = copy.deepcopy(feature_selector_params_sfm_default)
feature_selector_params_sfm["params"]["estimator"]["params"]["n_estimators"] = 250 # max = len(train_data.columns) - 1
feature_selector_params_sfm["params"]["threshold"] = "0*median"
feature_selection_info_sfm = select_feature(train_data, feature_selector_params_sfm)

feature_selection_infos = {
     "var" : feature_selection_info_var,
     "licor" : feature_selection_info_licor,
     "xicor" : feature_selection_info_xicor,
     "model" : feature_selection_info_sfm
}


--- 피처 선택기: FeatureFilter ---

--- 피처 필터링 시작 ---
    - 분산 필터링 후 남은 피처 수: 1401

피처 선택 결과가 'data/result/jsons\feature_selection_info_250901_222541_01e6e211.json' 파일에 저장되었습니다.

- 최종 피처 수: 1401

--- 피처 선택기: FeatureFilter ---

--- 피처 필터링 시작 ---
    - 타겟 선형 상관관계 필터링 후 남은 피처 수: 1650

피처 선택 결과가 'data/result/jsons\feature_selection_info_250901_222542_b3dbd5df.json' 파일에 저장되었습니다.

- 최종 피처 수: 1650

--- 피처 선택기: FeatureFilter ---

--- 피처 필터링 시작 ---
    - 타겟 Xi Cor 필터링 후 남은 피처 수: 1650

피처 선택 결과가 'data/result/jsons\feature_selection_info_250901_222543_14cdcea5.json' 파일에 저장되었습니다.

- 최종 피처 수: 1650

--- 피처 선택기: SFM ---
--- SFM 선택기 완료 ---
남은 피처 수: 1650

피처 선택 결과가 'data/result/jsons\feature_selection_info_250901_222545_b32881cb.json' 파일에 저장되었습니다.

- 최종 피처 수: 1650


In [21]:
for fileter_name, feature_selection_info in feature_selection_infos.items():
    # print(feature_selection_info["feature_selector_name"])
    # print(feature_selection_info["filter_methods"])
    # print(feature_selection_info["initial_feature_count"])
    # print(feature_selection_info["final_feature_count"])
    # print(feature_selection_info["feature_selection_info_json_path"])
    
    print("# of feature:", feature_selection_info["final_feature_count"], ",  filter: ", feature_selection_info["feature_selector_name"], fileter_name)


# of feature: 1401 ,  filter:  FeatureFilter var
# of feature: 1650 ,  filter:  FeatureFilter licor
# of feature: 1650 ,  filter:  FeatureFilter xicor
# of feature: 1650 ,  filter:  SFM model


In [22]:
# 피처 선택기 중요도 정보를 서머리

features_values_dfs = pd.DataFrame(
    list(train_data.columns), 
    columns=['feature_name']
)

for fileter_name, feature_selection_info in feature_selection_infos.items():
    if feature_selection_info["feature_selector_name"] == 'FeatureFilter':
        if feature_selection_info["filter_methods"]["apply_variance_filter"] == True:
            value_type = "variance"
        elif feature_selection_info["filter_methods"]["apply_target_linear_corr_filter"] == True:
            value_type = "target_linear_correlation"
        elif feature_selection_info["filter_methods"]["apply_target_xicor_filter"] == True:
            value_type = "target_xicor_correlation"
        features_values = feature_selection_info["selection_details"][value_type]["features_values_checked"]
    else : 
        value_type = "importances"
        features_values = feature_selection_info["selection_details"][value_type]

    features_values_df = pd.DataFrame(
        list(features_values.items()), 
        columns=['feature_name', feature_selection_info["feature_selector_name"]+"_"+value_type]
    )

    features_values_dfs = pd.merge(
        features_values_dfs,
        features_values_df,
        how='outer',
        left_on='feature_name',
        right_on='feature_name'
        )

In [23]:
# # 변수선택 방법별 변수선택 결과 데이터 : dic

# feature_selection_info

In [24]:
# # 변수선택 방법별 변수선택 결과 요약 데이터 : df

# features_values_dfs

In [25]:
# feature select test : pipeline
## 선택한 피처셋 feature_selection_info['final_features'] 리스트를 모델링 파이프라인에 적용, 모델링 결과를 리턴

def pl_fs_test(
        train_data,
        feature_selection_info,
        train_parameters,
        test_data
        ):
    
    trained_model, feature_importance, train_parameters_info = train_model_rf_cv(train_data.copy(), feature_selection_info, train_parameters)
    forecast_dataset, shap_values_random_forest = forecast(test_data, trained_model, feature_selection_info)
    train_dataset_proba, best_threshold = find_best_threshold(trained_model, train_data.copy(), feature_selection_info)
    roc_data, auc_score = roc_from_scratch(forecast_dataset, test_data, partitions=100)
    train_dataset_metrics = create_metrics_on_train(train_dataset_proba, best_threshold)
    metrics = create_metrics(forecast_dataset, test_data, auc_score, best_threshold)
    results = create_results(forecast_dataset, test_data, best_threshold)
    
    return \
        trained_model, \
        feature_importance, \
        forecast_dataset, \
        train_dataset_proba, \
        best_threshold, \
        roc_data, \
        auc_score, \
        train_dataset_metrics, \
        metrics, \
        results

##### 사용자가 지정한 임계값 활용 - 변수선택 > 성능체크 : 단순한 모델 파이프라인 활용

In [26]:
# 지정한 임계값 기준 피처선택에 대한 모델적용 및 결과 정리
# feature select test - submit and summary result

usr_pl_fs_test_result_ftpn_df = pd.DataFrame()
usr_pl_fs_test_result_features_values_dfs = pd.DataFrame(
    list(train_data.columns), 
    columns=['feature_name']
)

for fileter_name, feature_selection_info in feature_selection_infos.items():
    trained_model, \
    feature_importance, \
    forecast_dataset, \
    train_dataset_proba, \
    best_threshold, \
    roc_data, \
    auc_score, \
    train_dataset_metrics, \
    metrics, \
    results \
    = \
    pl_fs_test(
            train_data,
            feature_selection_info,
            train_parameters_list_default["rf_cv"],
            test_data
            )

    dict_ftpn = metrics["dict_ftpn"]
    
    new_row = {
        'fn': dict_ftpn.get('fn'),
        'fp': dict_ftpn.get('fp'),
        'tn': dict_ftpn.get('tn'),
        'tp': dict_ftpn.get('tp'),
        'feature_selector_name' : feature_selection_info["feature_selector_name"],
        'filter_methods' : feature_selection_info["filter_methods"],
        'initial_feature_count' : feature_selection_info["initial_feature_count"],
        'final_feature_count' : feature_selection_info["final_feature_count"],
        'feature_selection_info_json_path' : feature_selection_info["feature_selection_info_json_path"]
    }
    
    usr_pl_fs_test_result_ftpn_df = pd.concat([usr_pl_fs_test_result_ftpn_df, pd.DataFrame([new_row])], ignore_index=True)

    if feature_selection_info["feature_selector_name"] == 'FeatureFilter':
        if feature_selection_info["filter_methods"]["apply_variance_filter"] == True:
            value_type = "variance"
        elif feature_selection_info["filter_methods"]["apply_target_linear_corr_filter"] == True:
            value_type = "target_linear_correlation"
        elif feature_selection_info["filter_methods"]["apply_target_xicor_filter"] == True:
            value_type = "target_xicor_correlation"
        features_values = feature_selection_info["selection_details"][value_type]["features_values_checked"]
    else : 
        value_type = "importances"
        features_values = feature_selection_info["selection_details"][value_type]

    features_values_df = pd.DataFrame(
        list(features_values.items()), 
        columns=['feature_name', feature_selection_info["feature_selector_name"]+"_"+value_type]
    )

    usr_pl_fs_test_result_features_values_dfs = pd.merge(
        usr_pl_fs_test_result_features_values_dfs,
        features_values_df,
        how='outer',
        left_on='feature_name',
        right_on='feature_name'
        )

      Training the Random Forest model with cross-validation & hyperparameter tuning...

Fitting 3 folds for each of 18 candidates, totalling 54 fits

    Best parameters found: {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 50}
    Best F2 (class=1) score (CV): 0.2258

      Forecasting the test dataset...
      Forecasting done!
Best threshold for F2 score: 0.7172 with F2 score: 0.6153
      Calculation of the ROC curve...
      Calculation done
      Scoring...
      Scoring done

      Creating the metrics...
      Training the Random Forest model with cross-validation & hyperparameter tuning...

Fitting 3 folds for each of 18 candidates, totalling 54 fits

    Best parameters found: {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 20}
    Best F2 (class=1) score (CV): 0.2240

      Forecasting the test dataset...
      Forecasting done!
Best threshold for F2 score: 0.6970 with F2 score: 0.5909
      Calculation of the ROC curve...
      Calculation done
      Sco

In [27]:
# 피처선택셋 모델성능요약 - 사용자 피처선택 임계값 기준

usr_pl_fs_test_result_ftpn_df

,fn,fp,tn,tp,feature_selector_name,filter_methods,initial_feature_count,final_feature_count,feature_selection_info_json_path
0,5,180,707,13,FeatureFilter,"{'apply_variance_filter': True, 'var_threshold...",1650,1401,data/result/jsons\feature_selection_info_25090...
1,6,184,703,12,FeatureFilter,"{'apply_variance_filter': False, 'var_threshol...",1650,1650,data/result/jsons\feature_selection_info_25090...
2,6,184,703,12,FeatureFilter,"{'apply_variance_filter': False, 'var_threshol...",1650,1650,data/result/jsons\feature_selection_info_25090...
3,6,184,703,12,SFM,apply_SelectFromModel_filter,1650,1650,data/result/jsons\feature_selection_info_25090...


In [28]:
# 피처선택셋 모델성능요약 + 사용자 피처선택 임계값 적용 컬럼(참고용)

usr_pl_fs_test_result_features_values_dfs

,feature_name,FeatureFilter_variance,FeatureFilter_target_linear_correlation,FeatureFilter_target_xicor_correlation,SFM_importances
0,AC_COIL_FACTOR of 1103959_69_1133529_YPP,2.951574e-33,0.021211,0.259178,0.000000
1,AC_COIL_FACTOR of 1103959_69_1133529_cp1_cp1p5,2.951574e-33,0.021211,0.259178,0.000000
2,AC_COIL_FACTOR of 1103959_69_1133529_cp1_cp1p5...,2.951574e-33,0.021211,0.259178,0.000000
3,AC_COIL_FACTOR of 1103959_69_1133592_RPP,2.951574e-33,0.021211,0.259178,0.000000
4,AC_GAIN_32 of 1103959_69_1133529_cp1,0.000000e+00,NaN,0.456424,0.000000
...,...,...,...,...,...
1646,Zb_V_OBVOL_VSS_L of 1103959_69_1133529_cp1_cp1...,1.000277e+00,0.017764,0.503494,0.000395
1647,Zb_V_OBVOL_VSS_L of 1103959_69_1133529_cp1p5,1.000277e+00,0.026030,0.378215,0.000935
1648,Zb_V_OBVOL_VSS_L of 1103959_69_1133592_QPP,1.000277e+00,0.001384,0.467744,0.000143
1649,Zb_V_OBVOL_VSS_L of 1103959_69_JPP,1.000277e+00,0.067453,0.570805,0.000389


#####  피처 중요도 정보활용 - 최적선택 > 성능체크 : 단순한 모델 파이프라인 활용

###### 최적선택 방법은 XGBClassifier 모델, GridSearchCV 최적값 서칭 사용

In [29]:
feature_importance_df = features_values_dfs.copy()

In [30]:
### 적용

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import fbeta_score, make_scorer, confusion_matrix
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
import uuid

In [31]:
# train_data

In [32]:
# target은 가장 오른쪽 열을 선택
target_df = train_data.iloc[:, -1]
# feature는 나머지 모든 열
features_df = train_data.iloc[:, :-1]

X_train, X_test, y_train, y_test = train_test_split(features_df, target_df, test_size=0.2, random_state=42)

In [33]:
# run_optimization_for_feature_importance : 특정 중요도 컬럼을 기준으로 피처를 선택하고 최적의 모델을 찾는 함수

def run_optimization_for_feature_importance(train_data, target_data, feature_importance_df, importance_column, k_percentiles):
    """
    특정 중요도 컬럼을 기준으로 피처를 선택하고 최적의 모델을 찾는 함수
    
    Parameters:
    - train_data (pd.DataFrame): 훈련 데이터
    - target_data (pd.Series): 타겟 데이터
    - feature_importance_df (pd.DataFrame): 피처 중요도 정보가 담긴 DataFrame
    - importance_column (str): 중요도 순위를 결정할 컬럼명
    - k_percentiles (list): 선택할 피처의 백분위수 후보 리스트 (예: [0.05, 0.1, 0.25, 0.5])

    Returns:
    - pd.DataFrame: 최적화된 피처 중요도 정보
    - pd.DataFrame: 모델 성능 요약 정보
    """
    f2_scorer = make_scorer(fbeta_score, beta=2.0)
    
    # 중요도 컬럼의 값에 따라 상위 K개의 피처를 선택
    sorted_features = feature_importance_df.sort_values(
        by=importance_column, ascending=False
    )['feature_name']
    
    # 백분위수를 실제 피처 개수로 변환
    n_features_total = len(sorted_features)
    k_values = [max(1, int(n_features_total * p)) for p in k_percentiles]
    
    best_k = k_values[0]
    best_score = -1.0
    best_pipeline = None
    selected_feature_list = []
    # ⭐️ 최적의 백분위수 값을 저장할 변수
    best_k_percentile = k_percentiles[0]

    for i, k in enumerate(k_values):
        top_k_features = sorted_features.head(k).tolist()
        
        # 최적 피처로만 구성된 데이터셋 준비
        X_train_filtered = train_data[top_k_features]
        
        # 모델 훈련 파이프라인 (SelectKBest 대신 피처 직접 선택)
        pipeline = Pipeline([
            ('model', XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss'))
        ])
        
        param_grid = {
            'model__n_estimators': [50, 100],
            'model__max_depth': [3, 5]
        }
        
        grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring=f2_scorer, n_jobs=-1)
        grid_search.fit(X_train_filtered, target_data)

        # 현재 K의 성능 평가
        if grid_search.best_score_ > best_score:
            best_score = grid_search.best_score_
            best_k = k
            best_pipeline = grid_search.best_estimator_
            selected_feature_list = top_k_features
            # ⭐️ 최적의 백분위수 업데이트
            best_k_percentile = k_percentiles[i]

    # 최적 모델의 피처 중요도 및 성능 정보 생성
    best_xgb_model = best_pipeline.named_steps['model']
    
    feature_info = pd.DataFrame({
        'feature_name': train_data.columns
    })
    feature_info['is_selected'] = feature_info['feature_name'].isin(selected_feature_list)
    feature_info['importance_column'] = importance_column
    feature_importances = {name: 0 for name in train_data.columns}
    
    # 선택된 피처에 대해서만 중요도 점수를 부여
    for i, importance in enumerate(best_xgb_model.feature_importances_):
        if i < len(selected_feature_list):
            feature_importances[selected_feature_list[i]] = importance
        
    feature_info['importance_score'] = feature_info['feature_name'].map(feature_importances)
    feature_info['feature_value_by_importance_column'] = feature_info['feature_name'].map(
        feature_importance_df.set_index('feature_name')[importance_column]
    )

    # 테스트 데이터로 최종 성능 평가
    y_pred = best_pipeline.predict(X_test[selected_feature_list])
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    performance_summary = pd.DataFrame([{
        'fn': fn,
        'fp': fp,
        'tn': tn,
        'tp': tp,
        'feature_selector_name': importance_column,
        'initial_feature_count': X_train.shape[1],
        'final_feature_count': len(selected_feature_list),
        'f2_score': fbeta_score(y_test, y_pred, beta=2.0),
        'importance_column': importance_column,
        # ⭐️ 최적 백분위수 컬럼 추가
        'best_k_percentile': best_k_percentile
    }])

    return feature_info, performance_summary

In [ ]:
feature_importance_df

In [34]:
# 중요도 컬럼별로 최적화 반복 수행 및 결과 누적
# 피처이름 컬럼 제외 2번째 컬럼 이후의 중요도 컬럼명을 리스트로 저장
importance_cols = feature_importance_df.columns[1:].tolist()

# ⭐️ 백분위수 후보 리스트로 변경 : 사용자가 정의함
k_percentiles = [0.05, 0.1, 0.25, 0.5, 0.75, 0.8, 0.85, 0.9]

all_feature_infos = []
all_performance_summaries = []

for col in importance_cols:
    print(f"\n--- {col} 컬럼 기준 최적화 수행 ---")
    feat_info, perf_summary = run_optimization_for_feature_importance(
        X_train, y_train, feature_importance_df, col, k_percentiles
    )
    all_feature_infos.append(feat_info)
    all_performance_summaries.append(perf_summary)

final_feature_info_df = pd.concat(all_feature_infos, ignore_index=True)
final_feature_performance_summary_df = pd.concat(all_performance_summaries, ignore_index=True)

# 4. 결과 출력 및 CSV 저장
print("\n--- 최종 누적된 피처 중요도 정보 ---")
print(final_feature_info_df.head(10))
final_feature_info_df.to_csv('final_feature_info.csv', index=False)

print("\n--- 최종 누적된 모델 성능 요약 ---")
print(final_feature_performance_summary_df)
final_feature_performance_summary_df.to_csv('final_feature_performance_summary.csv', index=False)


--- FeatureFilter_variance 컬럼 기준 최적화 수행 ---

--- FeatureFilter_target_linear_correlation 컬럼 기준 최적화 수행 ---

--- FeatureFilter_target_xicor_correlation 컬럼 기준 최적화 수행 ---

--- SFM_importances 컬럼 기준 최적화 수행 ---

--- 최종 누적된 피처 중요도 정보 ---
                                        feature_name  is_selected  \
0                                                  X         True   
1                                                  Y         True   
2                                             Radius         True   
3           AC_COIL_FACTOR of 1103959_69_1133529_YPP        False   
4     AC_COIL_FACTOR of 1103959_69_1133529_cp1_cp1p5        False   
5  AC_COIL_FACTOR of 1103959_69_1133529_cp1_cp1p5...        False   
6           AC_COIL_FACTOR of 1103959_69_1133592_RPP        False   
7               AC_GAIN_32 of 1103959_69_1133529_cp1        False   
8     AC_GAIN_32 of 1103959_69_1133529_cp1_cp1p5_YPP        False   
9  AC_GAIN_32 of 1103959_69_1133529_cp1_cp1p5_YPP...        False   

        

In [35]:
# 자동(최적) 피처선택 결과 데이터 : df
final_feature_info_df

,feature_name,is_selected,importance_column,importance_score,feature_value_by_importance_column
0,X,True,FeatureFilter_variance,0.000000,1.000277e+00
1,Y,True,FeatureFilter_variance,0.005631,1.000277e+00
2,Radius,True,FeatureFilter_variance,0.001125,1.000277e+00
3,AC_COIL_FACTOR of 1103959_69_1133529_YPP,False,FeatureFilter_variance,0.000000,2.951574e-33
4,AC_COIL_FACTOR of 1103959_69_1133529_cp1_cp1p5,False,FeatureFilter_variance,0.000000,2.951574e-33
...,...,...,...,...,...
6595,FAILING_BINOUTS(1) of 1103959_69_JPP_GbGbSbSbPoPo,False,SFM_importances,0.000000,7.015850e-05
6596,FAILING_BINOUTS(1) of 1103959_69_JPP_GbPoPo,False,SFM_importances,0.000000,0.000000e+00
6597,FAILING_BINOUTS(1) of 1103959_69_JPP_PoPo,False,SFM_importances,0.000000,6.139330e-07
6598,FAILING_BINOUTS(1) of 1103959_69_JPP_SbPoPo,False,SFM_importances,0.000000,0.000000e+00


In [36]:
# 자동(최적) 피처선택 결과 데이터 요약 : df
final_feature_performance_summary_df

,fn,fp,tn,tp,feature_selector_name,initial_feature_count,final_feature_count,f2_score,importance_column,best_k_percentile
0,14,7,703,0,FeatureFilter_variance,1650,1238,0.0,FeatureFilter_variance,0.75
1,14,7,703,0,FeatureFilter_target_linear_correlation,1650,165,0.0,FeatureFilter_target_linear_correlation,0.10
2,14,6,704,0,FeatureFilter_target_xicor_correlation,1650,1485,0.0,FeatureFilter_target_xicor_correlation,0.90
3,14,6,704,0,SFM_importances,1650,825,0.0,SFM_importances,0.50


#####  피처 중요도 정보활용 - 최적선택 > 성능체크 : 단순한 모델 파이프라인 활용

In [37]:
# 성능체크파이프라인 입력 준비 : feature_selection_results

feature_selection_results = {}

# Iterate line by line
for idx, row in final_feature_performance_summary_df.iterrows():
    feature_selection_result = {}
    feature_selection_result["feature_selector_idx"] = idx
    feature_selection_result["feature_selector_name"] = row['feature_selector_name']
    feature_selection_result["initial_feature_count"] = row['initial_feature_count']
    feature_selection_result["final_feature_count"] = row['final_feature_count']
    feature_selection_result.setdefault("Params", {})["f2_score"] = row['f2_score']
    feature_selection_result.setdefault("Params", {})["best_k_percentile"] = row['best_k_percentile']
    
    feature_name_list = final_feature_info_df[
        (final_feature_info_df["importance_column"] == row['feature_selector_name']) &
        (final_feature_info_df["is_selected"] == True)
    ]["feature_name"].tolist()
    feature_selection_result["final_features"] = feature_name_list

    feature_selection_results[idx] = feature_selection_result

In [38]:
# feature_selection_results

In [39]:
# 최적피처셋 성능체크
# feature select test - submit and summary result

opt_pl_fs_test_result_ftpn_df = pd.DataFrame()
opt_pl_fs_test_result_features_values_dfs = pd.DataFrame(
    list(train_data.columns), 
    columns=['feature_name']
)

feature_importance_set = pd.DataFrame()

for fileter_name, feature_selection_info in feature_selection_results.items():
    trained_model, \
    feature_importance, \
    forecast_dataset, \
    train_dataset_proba, \
    best_threshold, \
    roc_data, \
    auc_score, \
    train_dataset_metrics, \
    metrics, \
    results \
    = \
    pl_fs_test(
            train_data,
            feature_selection_info,
            train_parameters_list_default["rf_cv"],
            test_data
            )

    dict_ftpn = metrics["dict_ftpn"]
    
    new_row = {
        'fn': dict_ftpn.get('fn'),
        'fp': dict_ftpn.get('fp'),
        'tn': dict_ftpn.get('tn'),
        'tp': dict_ftpn.get('tp'),
        'feature_selector_idx' : feature_selection_info["feature_selector_idx"],
        'feature_selector_name' : feature_selection_info["feature_selector_name"],
        'initial_feature_count' : feature_selection_info["initial_feature_count"],
        'final_feature_count' : feature_selection_info["final_feature_count"],
        'final_feature_selector_f2score' : feature_selection_info["Params"]["f2_score"],
        'final_feature_selector_best_k_percentile' : feature_selection_info["Params"]["best_k_percentile"]
    }
    
    opt_pl_fs_test_result_ftpn_df = pd.concat([opt_pl_fs_test_result_ftpn_df, pd.DataFrame([new_row])], ignore_index=True)
    features_values = feature_importance[["Features", "Importance"]].rename(columns={"Importance": "Importance(model)"+"_"+feature_selection_info["feature_selector_name"] })

    opt_pl_fs_test_result_features_values_dfs = pd.merge(
        opt_pl_fs_test_result_features_values_dfs,
        features_values,
        how='left',
        left_on='feature_name',
        right_on='Features'
    ).drop('Features', axis=1) # The .drop() method is used to drop the column

      Training the Random Forest model with cross-validation & hyperparameter tuning...

Fitting 3 folds for each of 18 candidates, totalling 54 fits

    Best parameters found: {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 20}
    Best F2 (class=1) score (CV): 0.2206

      Forecasting the test dataset...
      Forecasting done!
Best threshold for F2 score: 0.6566 with F2 score: 0.5736
      Calculation of the ROC curve...
      Calculation done
      Scoring...
      Scoring done

      Creating the metrics...
      Training the Random Forest model with cross-validation & hyperparameter tuning...

Fitting 3 folds for each of 18 candidates, totalling 54 fits

    Best parameters found: {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 20}
    Best F2 (class=1) score (CV): 0.1966

      Forecasting the test dataset...
      Forecasting done!
Best threshold for F2 score: 0.7980 with F2 score: 0.5182
      Calculation of the ROC curve...
      Calculation done
      Sco

In [40]:
opt_pl_fs_test_result_ftpn_df

,fn,fp,tn,tp,feature_selector_idx,feature_selector_name,initial_feature_count,final_feature_count,final_feature_selector_f2score,final_feature_selector_best_k_percentile
0,4,193,694,14,0,FeatureFilter_variance,1650,1238,0.0,0.75
1,8,199,688,10,1,FeatureFilter_target_linear_correlation,1650,165,0.0,0.10
2,8,165,722,10,2,FeatureFilter_target_xicor_correlation,1650,1485,0.0,0.90
3,8,132,755,10,3,SFM_importances,1650,825,0.0,0.50


In [41]:
opt_pl_fs_test_result_features_values_dfs

,feature_name,Importance(model)_FeatureFilter_variance,Importance(model)_FeatureFilter_target_linear_correlation,Importance(model)_FeatureFilter_target_xicor_correlation,Importance(model)_SFM_importances
0,X,0.000000,NaN,NaN,0.000000
1,Y,0.003681,NaN,0.016581,NaN
2,Radius,0.031251,NaN,0.000000,0.006605
3,AC_COIL_FACTOR of 1103959_69_1133529_YPP,NaN,NaN,NaN,NaN
4,AC_COIL_FACTOR of 1103959_69_1133529_cp1_cp1p5,NaN,NaN,NaN,NaN
...,...,...,...,...,...
1646,FAILING_BINOUTS(1) of 1103959_69_JPP_GbPoPo,NaN,NaN,0.000000,NaN
1647,FAILING_BINOUTS(1) of 1103959_69_JPP_PoPo,NaN,NaN,0.000000,NaN
1648,FAILING_BINOUTS(1) of 1103959_69_JPP_SbPoPo,NaN,NaN,0.000000,NaN
1649,band gap dpat_ok for band gap,NaN,NaN,0.000000,NaN
